In [ ]:
import gzip
import pyshark

# Step 1: Decompress the .pcap.gz file
def decompress_gz(gz_file, output_file):
    with gzip.open(gz_file, 'rb') as f_in:
        with open(output_file, 'wb') as f_out:
            f_out.write(f_in.read())

# Step 2: Analyze the decompressed PCAP file
def analyze_pcap(pcap_file):
    # Read the pcap file using pyshark
    cap = pyshark.FileCapture(pcap_file)
    
    # Display general packet statistics
    print(f"Total packets: {len(cap)}")
    
    # Example: Analyze first few packets
    for packet in cap[:10]:  # Get the first 10 packets
        print(packet)  # Print packet details
    
    # Filter specific protocols (e.g., HTTP, DNS, TCP, etc.)
    http_packets = [pkt for pkt in cap if 'HTTP' in pkt]
    print(f"Total HTTP packets: {len(http_packets)}")
    
    # Close the capture after use
    cap.close()

# Main execution
if __name__ == "__main__":
    gz_file = '/home/shyam/jupy/ddos_scrubber/data/amppot_dataset/amppot-ynu_sensor009_20210609.pcap.gz'
    pcap_file = '/home/shyam/jupy/ddos_scrubber/data/amppot_dataset/amppot-ynu_sensor009_20210609.pcap'
    
    # Step 1: Decompress
    decompress_gz(gz_file, pcap_file)
    
    # Step 2: Analyze the PCAP
#     analyze_pcap(pcap_file)


In [238]:
# Read amppot file
import pandas as pd
filename = "/home/shyam/phd-work/DDOS/amppot-data/amppot-2024-12-01.csv.gz"
# Define custom headers
column_names = ['hpdate', 'dport', 'src', 't_start', 't_end', 'packets', 'hpids']

# Read the CSV file without headers and assign column names
df = pd.read_csv(filename, compression='gzip', header=None, names=column_names)

# Convert to datetime
df['t_start'] = pd.to_datetime(df['t_start'])
df['t_end'] = pd.to_datetime(df['t_end'])
df['hpdate'] = pd.to_datetime(df['hpdate']).dt.date

# Filter: duration > 5 minutes AND hpdate == 2024-12-03
filtered_df = df[
    (df['t_end'] - df['t_start'] > pd.Timedelta(minutes=5)) &
    (df['hpdate'] == pd.to_datetime('2024-12-03').date()) &
    (df['t_end'] - df['t_start'] < pd.Timedelta(minutes=60))
]

# Show result
filtered_df

,hpdate,dport,src,t_start,t_end,packets,hpids
14743,2024-12-03,17,45.145.166.121,2024-12-03 12:14:19,2024-12-03 12:30:11,3692,9
14751,2024-12-03,19,85.215.114.192,2024-12-03 04:11:18,2024-12-03 04:16:28,1236,8
14762,2024-12-03,53,1.157.212.139,2024-12-03 09:20:12,2024-12-03 09:31:59,116,1
14855,2024-12-03,53,2.83.190.221,2024-12-03 00:23:32,2024-12-03 00:47:19,184,1
14863,2024-12-03,53,2.217.184.247,2024-12-03 09:44:52,2024-12-03 10:10:33,116,1
...,...,...,...,...,...,...,...
22797,2024-12-03,1900,1.32.226.28,2024-12-03 01:23:17,2024-12-03 02:03:16,200,2
22817,2024-12-03,1900,103.194.104.234,2024-12-03 12:37:33,2024-12-03 12:43:29,2149,8
22821,2024-12-03,1900,110.40.136.66,2024-12-03 10:26:07,2024-12-03 11:09:10,2162,8
22840,2024-12-03,11211,111.38.40.6,2024-12-03 08:39:14,2024-12-03 08:48:03,3740,2


In [65]:
unique_prefixes_df = df["src"].unique()
len(unique_prefixes_df)

30687

In [239]:
filtered_df[filtered_df["src"].str.contains("66.118")]

,hpdate,dport,src,t_start,t_end,packets,hpids
15800,2024-12-03,53,66.118.234.91,2024-12-03 00:39:11,2024-12-03 01:31:04,193,1


In [86]:
from datetime import datetime, timezone

# GMT time as a string
gmt_time_str = "2024-12-03 12:14:19"

# Parse the string as a datetime object with UTC timezone (GMT == UTC)
dt = datetime.strptime(gmt_time_str, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)

# Convert to Unix timestamp (seconds since epoch)
unix_timestamp = int(dt.timestamp())

print(f"Unix timestamp: {unix_timestamp}")


Unix timestamp: 1733228059


In [22]:
# Check BGP updates (announcements) for a few targeted IP addresses in amppot file
import pybgpstream
import datetime
from datetime import datetime, timedelta, timezone
import csv

ips = filtered_df["src"]
start_times = filtered_df["t_start"]
end_times = filtered_df["t_end"]


for idx, ip in enumerate(ips[389:391], start=389):
    prefix = str(ip) +"/32"

    # Convert to datetime object
    # Parse the string as a datetime object with UTC timezone (GMT == UTC)
    dt = datetime.strptime(str(start_times.iloc[idx]), "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    posix_start_time = int(dt.timestamp())
    
    dt = datetime.strptime(str(end_times.iloc[idx]), "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)
    posix_end_time = int(dt.timestamp())
    
    from_time =  posix_start_time - 600 # - 600 for 10 minutes, -7200 for 2 hours early
    until_time = posix_end_time + 600 # +600 for 10 minutes, + 7200 for 2 hours later

    # Convert to GMT time
    from_time_utc = datetime.utcfromtimestamp(from_time)
    until_time_utc = datetime.utcfromtimestamp(until_time)

    # Convert UTC datetime to string
    from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
    until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

#     print("Time %s and %s" %(from_time_utc_str, until_time_utc_str))


    stream = pybgpstream.BGPStream(
        from_time=from_time_utc_str, 
        until_time=until_time_utc_str,
        record_type="updates", # By default it is Update announcement 
        project="ris",
        filter = "prefix less "+prefix
        )
    stream.set_data_interface_option("broker", "cache-dir", "/home/shyam/jupy/cache")

    p = [] # List containing immediate provider
    o = [] # Origin ASN

#     print("Extracting records for prefix %s" %prefix)


    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        for elem in rec:
            # Find second ASN in an AS path
            
            if elem.type == "A":
                as_path = elem.fields["as-path"]
                pfx = elem.fields["prefix"]

                # Convert as_path into list
                as_path_list = as_path.split()
                orig = as_path_list[-1]
            else:
                as_path_list = []
                orig = []
                prefix = "" # For some updates with had error state e.g. U|S| instead of A or W
            # List of 28 confirmed scrubbers from bgp.tools ddosm tags and DDoScovery paper.
            scrubbers = ['32787', '13335', '19551', '19905', '198949', '57724', '54113', '21859', '19324', 
                         '3223', '137409', '396998', '30456', '8757', '34309', '35280', '197068', 
                         '199524', '45474', '200020', '42649', '59796', '5405', '20052', ' 394009', '401073'] 

            single_data = []
            # Discard different forms of origins example AS-Set, confederation set/sequence. 
            # Take only a single AS origin which is common for a scrubbing activity
            matched_scrubber = set(scrubbers) & set(as_path_list)
            
            # Convert it into string
            scrubber = ','.join(matched_scrubber)
                                    
            if pfx != '0.0.0.0/0' and pfx!= '' and len(as_path_list) > 1 and scrubber:  # intersection is not empty
                
                # Find second ASN in an AS path
                second_as = as_path_list[-2]
                # Extract its upstream provider to check if that one contains an scrubbers' ASN
                single_data.append({"prefix": pfx})
                single_data.append({"provider": second_as})
                single_data.append({"origin": orig})
                single_data.append({"time": time})
                single_data.append({"as_path": as_path_list})
                single_data.append({"scrubber": scrubber})
                p.append(single_data)
    
    providers = [entry[1]['provider'] for entry in p]
    # Find number of peers that are listed as DDoS mitigation 
    matching_elements = set(providers) & set(scrubbers)
#     print("Total matching ASes ",len(matching_elements))
    if (scrubber) and len(p) >0:
        # Save results into a csv file.
        # Flatten each list of single-key dictionaries into a single dictionary
        flattened_data = [{k: v for d in entry for k, v in d.items()} for entry in p]
        # Write to CSV
        with open('output_'+str(idx)+'.csv', 'a', newline='') as csvfile:
            fieldnames = ['prefix', 'provider', 'origin', 'time', 'as_path', 'scrubber']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

            writer.writeheader()
            for row in flattened_data:
                writer.writerow(row)
        print("Written to a file: output_"+str(idx)+".csv")
    print("Done for prefix %s" %prefix)
print("Completed.")

Written to a file: output_389.csv
Done for prefix 104.21.85.3/32
Completed.


In [23]:
# Merge all those output_<index>.csv files into a single file.
import pandas as pd
import glob

# Find all CSV files matching the pattern
csv_files = glob.glob("output/output_*.csv")

# Read and concatenate all files
df_list = [pd.read_csv(file) for file in csv_files]
merged_df = pd.concat(df_list, ignore_index=True)

# Save to a new CSV file
merged_df.to_csv("output_dec1_8_2024/merged_output.csv", index=False)
            

In [221]:
merged_df["provider"] = merged_df["provider"].astype("string")
merged_df["origin"] = merged_df["origin"].astype("string")


In [223]:
merged_df.to_csv("output_dec1_8_2024/merged_output.csv", index=False)


In [225]:
# Cases where a scrubber appears as a provider
a = merged_df[merged_df["scrubber"] == merged_df["provider"]]
len(a["prefix"].unique())

12

In [226]:
# Cases where a scrubber appears as an origin
b = merged_df[merged_df["scrubber"] == merged_df["origin"]]
len(b["prefix"].unique())

6

In [240]:
a = merged_df[merged_df["provider"] == "13335"]
a.sort_values(by='time')

,Unnamed: 0,prefix,provider,origin,time,as_path,scrubber
6,24,66.118.232.0/22,13335,399244,1733188351,"['13030', '13335', '399244']",13335
7,25,66.118.232.0/22,13335,399244,1733188989,"['6830', '13335', '399244']",13335
43,73,66.118.232.0/22,13335,399244,1733190997,"['13030', '13335', '399244']",13335
32,52,103.60.148.0/22,13335,132839,1733193012,"['46997', '38008', '13335', '132839']",13335
33,53,103.60.148.0/22,13335,132839,1733193464,"['38008', '13335', '132839']",13335
83,115,45.207.224.0/19,13335,139646,1733210513,"['212483', '13335', '139646']",13335
84,116,45.207.224.0/19,13335,139646,1733210513,"['212483', '13335', '139646']",13335
85,117,45.207.224.0/19,13335,139646,1733210584,"['212483', '13335', '139646']",13335
86,118,45.207.224.0/19,13335,139646,1733210584,"['212483', '13335', '139646']",13335
87,119,45.207.224.0/19,13335,139646,1733210675,"['28910', '31133', '13335', '139646']",13335
